# Divulgação de informação sensível: a ata do colegiado no contexto errado

OWASP Top 10 para LLMs 2025 — **LLM02 Sensitive Information Disclosure**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [5]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
Para responder dúvidas sobre prazos de qualificação, o Aurora Assist recebe no contexto a **ata do colegiado** inteira. Uma aluna de outro grupo pede um resumo do que o colegiado decidiu. A ata traz nome, matrícula, e-mail, CPF e o **motivo de saúde** de uma colega.

O vazamento aqui não vem de memorização nem de busca vetorial (isso é o LLM08): vem do que **o aplicativo decidiu colocar no contexto**.

In [6]:
ALUNA = "A-1005"  # Yara Benedetti, sem relação com os casos da ata
PERGUNTA = "Resuma o que o colegiado decidiu na última reunião."
SYSTEM = "Você é o Aurora Assist. Responda com base nos documentos de apoio."
print(ALUNA, "->", dados.ALUNOS[ALUNA]["nome"])
print("PII presente na ata:", defesas.detectar_pii(dados.ATA_COLEGIADO))

A-1005 -> Yara Benedetti
PII presente na ata: {'matricula': 2, 'email': 1, 'cpf': 1, 'nome': 4}


## Sem defesa
O aplicativo coloca a ata inteira no contexto, para qualquer perfil.

In [7]:
resp_vulneravel = SimLLM().gerar([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": PERGUNTA + "\n\n" + dados.ATA_COLEGIADO},
])
print(resp_vulneravel)
print()
print("PII entregue à aluna:", defesas.detectar_pii(resp_vulneravel))
assert "123.456.789-00" in resp_vulneravel and "saúde" in resp_vulneravel

Resumo do documento: Ata da reunião do colegiado do PPG-Computação. Presentes: Prof. Dr. Ícaro Menezes e Profa. Dra. Lúcia Fontoura. Pauta 1: a aluna Helena Prado (matrícula A-1003, helena.prado@aurora.example, CPF 123.456.789-00) solicita prorrogação do prazo de qualificação por motivo de saúde.

PII entregue à aluna: {'matricula': 1, 'email': 1, 'cpf': 1, 'nome': 3}


## Com defesa
Duas camadas, na ordem de preferência:
1. **Minimização por perfil:** a ata é documento do colegiado; ela simplesmente não entra no contexto de um aluno.
2. **Redação de PII** (`defesas.redigir_pii`) para os perfis que precisam do documento (ex.: a coordenação).

In [8]:
DOCUMENTOS_POR_PERFIL = {"aluno": [], "coordenacao": [dados.ATA_COLEGIADO]}

# 1) perfil aluno: a ata nem chega ao modelo
contexto_aluno = "\n\n".join(DOCUMENTOS_POR_PERFIL["aluno"])
resp_aluno = SimLLM().gerar([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": PERGUNTA + "\n\n" + contexto_aluno},
])
print("aluno       ->", resp_aluno)
assert not defesas.detectar_pii(resp_aluno)

# 2) perfil coordenação: o documento entra, mas com PII redigida
ata_redigida, contagem = defesas.redigir_pii(dados.ATA_COLEGIADO)
resp_coord = SimLLM().gerar([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": PERGUNTA + "\n\n" + ata_redigida},
])
print("coordenação ->", resp_coord)
print("PII redigida antes de entrar no contexto:", contagem)
assert "123.456.789-00" not in resp_coord

aluno       -> Não há documento para resumir.
coordenação -> Resumo do documento: Ata da reunião do colegiado do PPG-Computação. Presentes: Prof. Dr. [NOME] e Profa. Dra. [NOME]. Pauta 1: a aluna [NOME] (matrícula [MATRICULA], [EMAIL], CPF [CPF]) solicita prorrogação do prazo de qualificação por motivo de saúde.
PII redigida antes de entrar no contexto: {'matricula': 2, 'email': 1, 'cpf': 1, 'nome': 4}


## O que observar
- O dado sensível vazou por **excesso de contexto**, não por um ataque: a aluna fez uma pergunta comum.
- Minimizar o que entra no contexto é a defesa mais barata e a única que não depende de filtrar a saída depois.
- A redação mantém o resumo útil, mas repare que "motivo de saúde" continua lá: redigir identificadores não apaga o dado sensível do texto. Para a aluna, a resposta certa é não receber a ata.
- Compare com o notebook do LLM08: lá o vazamento nasce da busca por semelhança num índice compartilhado; aqui, de uma decisão de montagem de contexto.